# Event Recommendation Agent — Demo

This notebook demonstrates the LangGraph-based event recommendation agent: natural-language query understanding, a happy-path run, a fallback scenario, observability output, and a few example queries across different users.

## 1. Setup

Import the agent and load an example user profile.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from agent.recommendation_agent import invoke

with open(Path.cwd().parent / "data" / "users.json") as f:
    users = json.load(f)

example_user = users[0]
example_user

{'id': 'u01',
 'name': 'Alex Nguyen',
 'genres': ['Food & Drink', 'Sports', 'Film'],
 'past_events': ['e063', 'e090', 'e028', 'e145'],
 'avg_ticket_price': 50}

## 2. Natural-language query understanding

The agent parses free-text requests like *"find me a concert next week"* into structured intent (genres + a date range) before searching. It tries Amazon Bedrock first and falls back to deterministic keyword parsing if Bedrock isn't configured — both paths return the same shape, so nothing downstream needs to know which one ran.

In [2]:
result = invoke(example_user["id"], "Find me a concert for next week")
result["parsed_intent"]

2026-09-14 01:30:17,501 INFO agent.recommendation_agent: start_node: entering for user_id=u01 query='Find me a concert for next week'


2026-09-14 01:30:17,501 INFO agent.recommendation_agent: start_node: LLM path skipped (no AWS credentials configured)


2026-09-14 01:30:17,501 INFO agent.recommendation_agent: start_node finished in 0.52ms


2026-09-14 01:30:17,502 INFO agent.recommendation_agent: parse_query_node: entering with query='Find me a concert for next week'


2026-09-14 01:30:17,502 INFO agent.metrics: tool=parse_query[rule_based] latency_ms=0.19 success=True


...

2026-09-14 01:30:17,506 INFO agent.recommendation_agent: format_output_node: finalizing 5 recommendations via deterministic path (fallbacks used: ['alternative_source_fallback'])


2026-09-14 01:30:17,506 INFO agent.recommendation_agent: format_output_node finished in 0.12ms


2026-09-14 01:30:17,506 INFO agent.metrics: agent run complete: total_latency_ms=6.80 fallback_count=1


2026-09-14 01:30:17,506 INFO agent.recommendation_agent: invoke: completed for user_id=u01 via deterministic path in 6.80ms with 1 fallback(s)


{'genres': ['Music'], 'date_from': '2026-09-21', 'date_to': '2026-09-27'}

## 3. Structured output with latencies

In [3]:
print(f"User: {result['user_id']}")
print(f"Query: {result['query']}")
print(f"Parsed intent: {result['parsed_intent']}")
print(f"Total latency: {result['total_latency_ms']:.2f}ms")
print(f"Fallbacks used: {result['fallback_strategies_used']}")
print("\nRecommendations:")
for event in result["recommendations"]:
    print(f"  - {event['name']} ({event['genre']}) at {event['venue']} "
          f"on {event['date']} — ${event['price']}, popularity {event['popularity_score']}")

User: u01
Query: Find me a concert for next week
Parsed intent: {'genres': ['Music'], 'date_from': '2026-09-21', 'date_to': '2026-09-27'}
Total latency: 6.80ms
Fallbacks used: ['alternative_source_fallback']

Recommendations:
  - Indie Music Live (Music) at Skyline Rooftop on 2027-01-16 — $15, popularity 74
  - Local Music Tour Stop (Music) at Downtown Pavilion on 2026-12-12 — $0, popularity 72
  - Quiet Music Showcase (Music) at The Underground on 2026-09-17 — $20, popularity 71
  - Underground Music Jam (Music) at Lakeside Gardens on 2026-11-19 — $0, popularity 71
  - Grand Music Sessions (Music) at Central Park Bandshell on 2027-01-06 — $10, popularity 65


## 4. Trigger a fallback scenario

Passing an unknown user id forces the preference lookup to fail, which triggers `graceful_degradation_fallback` (Fallback A).

In [4]:
fallback_result = invoke("user-does-not-exist", "Recommend me an event")
print(f"Fallback count: {fallback_result['fallback_count']}")
print(f"Strategies used: {fallback_result['fallback_strategies_used']}")
print(f"Recommendations returned: {len(fallback_result['recommendations'])}")

2026-09-14 01:30:17,512 INFO agent.recommendation_agent: start_node: entering for user_id=user-does-not-exist query='Recommend me an event'


2026-09-14 01:30:17,513 INFO agent.recommendation_agent: start_node: LLM path skipped (no AWS credentials configured)


2026-09-14 01:30:17,513 INFO agent.recommendation_agent: start_node finished in 0.31ms


2026-09-14 01:30:17,513 INFO agent.recommendation_agent: parse_query_node: entering with query='Recommend me an event'


2026-09-14 01:30:17,513 INFO agent.metrics: tool=parse_query[rule_based] latency_ms=0.19 success=True


...

2026-09-14 01:30:17,517 INFO agent.recommendation_agent: format_output_node: finalizing 5 recommendations via deterministic path (fallbacks used: ['graceful_degradation_fallback'])


2026-09-14 01:30:17,517 INFO agent.recommendation_agent: format_output_node finished in 0.11ms


2026-09-14 01:30:17,517 INFO agent.metrics: agent run complete: total_latency_ms=5.04 fallback_count=1


2026-09-14 01:30:17,517 INFO agent.recommendation_agent: invoke: completed for user_id=user-does-not-exist via deterministic path in 5.04ms with 1 fallback(s)


Fallback count: 1
Strategies used: ['graceful_degradation_fallback']
Recommendations returned: 5


## 5. Metrics and logs

Every run captures per-tool latency and success/failure, plus any fallback events. The `execution_path` field says whether Claude on Bedrock chose the tool calls (`llm_tools`, tool names suffixed `[llm]`) or the deterministic path ran (`deterministic`, starting with `parse_query[rule_based]`). This notebook was executed without AWS credentials, so it shows the deterministic path.

In [5]:
print(result["logs"])
print()
print(fallback_result["logs"])

Tool calls:
  - parse_query[rule_based]: 0.19ms [OK]
  - fetch_user_preferences: 0.43ms [OK]
  - search_events: 0.49ms [OK]
  - rank_events: 0.39ms [OK]
Fallbacks triggered:
  - alternative_source_fallback
Total latency: 6.80ms
Fallback count: 1

Tool calls:
  - parse_query[rule_based]: 0.19ms [OK]
  - fetch_user_preferences: 0.63ms [FAILED]
  - search_events: 0.34ms [OK]
  - rank_events: 0.17ms [OK]
Fallbacks triggered:
  - graceful_degradation_fallback
Total latency: 5.04ms
Fallback count: 1


## 6. Example query variations

A few different phrasings — genre-only, time-only, and combined — across different users.

In [6]:
example_queries = [
    (users[1], "What should I see tonight?"),
    (users[2], "Find me a family event this month"),
    (users[3], "Any comedy shows this weekend?"),
]

for user, query in example_queries:
    r = invoke(user["id"], query)
    print(f"{user['name']} ({user['id']}): \"{query}\"")
    print(f"  parsed_intent={r['parsed_intent']}")
    print(f"  latency={r['total_latency_ms']:.2f}ms fallbacks={r['fallback_strategies_used']}")
    for event in r["recommendations"][:3]:
        print(f"    - {event['name']} ({event['genre']}) on {event['date']}")
    print()

2026-09-14 01:30:17,523 INFO agent.recommendation_agent: start_node: entering for user_id=u02 query='What should I see tonight?'


2026-09-14 01:30:17,524 INFO agent.recommendation_agent: start_node: LLM path skipped (no AWS credentials configured)


2026-09-14 01:30:17,524 INFO agent.recommendation_agent: start_node finished in 0.35ms


2026-09-14 01:30:17,524 INFO agent.recommendation_agent: parse_query_node: entering with query='What should I see tonight?'


2026-09-14 01:30:17,524 INFO agent.metrics: tool=parse_query[rule_based] latency_ms=0.13 success=True


...

2026-09-14 01:30:17,539 INFO agent.recommendation_agent: format_output_node: finalizing 5 recommendations via deterministic path (fallbacks used: ['alternative_source_fallback'])


2026-09-14 01:30:17,540 INFO agent.recommendation_agent: format_output_node finished in 0.11ms


2026-09-14 01:30:17,540 INFO agent.metrics: agent run complete: total_latency_ms=6.74 fallback_count=1


2026-09-14 01:30:17,540 INFO agent.recommendation_agent: invoke: completed for user_id=u04 via deterministic path in 6.74ms with 1 fallback(s)


Jordan Garcia (u02): "What should I see tonight?"
  parsed_intent={'genres': [], 'date_from': '2026-09-14', 'date_to': '2026-09-14'}
  latency=5.27ms fallbacks=['alternative_source_fallback']
    - Local Music Tour Stop (Music) on 2026-12-12
    - Quiet Music Showcase (Music) on 2026-09-17
    - Underground Music Jam (Music) on 2026-11-19

Taylor Smith (u03): "Find me a family event this month"
  parsed_intent={'genres': ['Family'], 'date_from': '2026-09-14', 'date_to': '2026-09-30'}
  latency=3.92ms fallbacks=[]
    - Live Family Fun Day (Family) on 2026-09-28

Morgan Patel (u04): "Any comedy shows this weekend?"
  parsed_intent={'genres': ['Comedy'], 'date_from': '2026-09-19', 'date_to': '2026-09-20'}
  latency=6.74ms fallbacks=['alternative_source_fallback']
    - Electric Comedy Standup Hour (Comedy) on 2026-12-22
    - Local Comedy Open Mic (Comedy) on 2026-09-25
    - Indie Comedy Night (Comedy) on 2026-11-25

